# **Bag of Words (BOW) Text Classification**

This notebook demonstrates how to use TFIDF Vectorizer from scikit-learn

_______________

### **Set-Up**

This section focuses on loading the data, python libraries (which will be used throughout this notebook)

In [1385]:
# Loading Libraries
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("--"*50)
print("Libraries loaded successfully.")
print("--"*50)

----------------------------------------------------------------------------------------------------
Libraries loaded successfully.
----------------------------------------------------------------------------------------------------


In [1386]:
# Loading the dataset
data = pd.read_csv("data/english_pages_metadata_clean_with_labels.csv")

print("--"*50)
print("First few Rows of the Dataset:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
First few Rows of the Dataset:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,both
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,both
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,both
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,both
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...",both
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,both
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,both
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,both
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,both


In [1387]:
data.drop(columns=["_merge"], inplace=True)

In [1388]:
# Dataset Shape:
print("--"*50)
print("Dataset Shape (Rows, Columns)")
print("--"*50)
data.shape

----------------------------------------------------------------------------------------------------
Dataset Shape (Rows, Columns)
----------------------------------------------------------------------------------------------------


(596, 6)

In [1389]:
# Dataset Information
print("--"*50)
print("Dataset Information")
print("--"*50)
data.info()

----------------------------------------------------------------------------------------------------
Dataset Information
----------------------------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   page_id             596 non-null    float64
 1   assigned_to         596 non-null    object 
 2   manual_label        596 non-null    object 
 3   manual_label_clean  596 non-null    object 
 4   manual_label_final  596 non-null    object 
 5   full_text           596 non-null    object 
dtypes: float64(1), object(5)
memory usage: 28.1+ KB


In [1390]:
# Identifying NULLs and NaNs in the data set
print("--"*50)
print("Null Values in the dataset")
print("--"*50)
print(data.isnull().sum())

----------------------------------------------------------------------------------------------------
Null Values in the dataset
----------------------------------------------------------------------------------------------------
page_id               0
assigned_to           0
manual_label          0
manual_label_clean    0
manual_label_final    0
full_text             0
dtype: int64


In [1391]:
print("--"*50)
print("Data Description for Numeric features:")
print("--"*50)
data.describe(include=[np.number])

----------------------------------------------------------------------------------------------------
Data Description for Numeric features:
----------------------------------------------------------------------------------------------------


,page_id
count,596.000000
mean,675.904362
std,449.207144
min,1.000000
25%,356.250000
50%,620.500000
75%,977.250000
max,2060.000000


Based on the above information, it is clear from a bird’s-eye view that there are no null or NaN (except for manual_label) values or even duplication of rows within the dataset.

____________

### **1. Text Preprocessing**

In this section, the primary focus is on cleaning the text data.Identifying and removing these characters early is important, as they can cause issues later in the pipeline and can negatively impact stability and compatibility. The primary goal here is to retain ASCII characters, such as English letters, numbers, and common punctuation (e.g., '(', ')', '[', ']', '+', '-', ' '), while ensuring that the existing structure of the text remains unchanged.

#### **1.1 Filtering out the English Words**

Here we will focusing on removing all the non-english character and preserve currance symbols.  `\x00-\x7f` - ASCII Range: It starts from ASCII 00 and ends at ASCII 127. Here the "\x" is an escape sequence telling the regex engine that the next two character are a hexadecimal, "00" means the start the first ASCII value and "7f" is the hexadecimal value for 127.

In [1392]:
# list of currency symbols (few of them might not be present, but still we will keep it)
currency_symbols = [
    # Paired (Multi-character) symbols (Longest first)
    'USD','AU$', 'C$', 'NZ$', 'HK$', 'S$', 'US$', 'NT$', 'MX$', 'R$', 'zł', 'Kč','₨',
    'kr', 'Ft', 'ден', 'р.', 'лв', '₡', '₢', '₣', '₥', 

    # Single character symbols (Specific before generic)
    '€', '£', '¥', '元', '₹', '฿', '₩', '₽', '₪', '₺', '₦', '₵', '₫', 
    '₱', '₭', '₮', '₼', '₸', '₴', '៛', '₲', '₡', '₾', '֏', '﷼', 'Ξ', 'Ł', '$'
]

curr_symbols = ''.join(re.escape(s) for s in currency_symbols)
# function to clean the full text column
def full_text_clean (text):
    #Regex Pattern to indentify all the ASCII characters while retaining currency symbols
    re_pattern = rf"[\x00-\x7F{curr_symbols}]+"

    clean = re.findall(re_pattern, text)

    return clean

In [1393]:
data['full_text_clean'] = data['full_text'].apply(full_text_clean)

print("--"*50)
print("New datastructure after cleaning the full_text columns:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after cleaning the full_text columns:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , A..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.2 Removal of Emails**

Removing email addresses because they are primarily alphanumeric in nature and do not carry meaningful semantic information. They also do not contribute as reliable features for text classification and may instead introduce noise into the model

In [1394]:
def extract_emails(text):
    """
    Removing all email addresses from text
    """
    # Regex pattern for emails
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
    clean_text = []
    
    # Find all matches
    for i in range (len(text)):
        clean_text.append(re.sub(email_pattern, '', text[i]))
    
    return clean_text

In [1395]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_emails)

print("--"*50)
print("Email Removal:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
Email Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , A..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.3 Removel of Phone Numbers**

Phone numbers do not carry meaningful semantic information for text classification, so it is better to remove them rather than retain them.

In [1396]:
def extract_phones(text):
    """
    Extract phone numbers in various formats
    Returns: list of phone numbers
    """
    clean_text = []
    
    # Pattern 1: 555-123-4567
    pattern1 = r'\b\d{3}-\d{3}-\d{4}\b'
    # Pattern 2: (555) 123-4567
    pattern2 = r'\(\d{3}\)\s*\d{3}-\d{4}'
    # Pattern 3: 800-555-0123 (toll-free)
    pattern3 = r'\b[8-9]00-\d{3}-\d{4}\b'
    # Pattern 4: +1-555-123-4567
    pattern4 = r'\+{1,3}-\d{3}-\d{3}-\d{4}'
    # Pattern 5 : +267 7610 0890
    pattern5 = r'\+\d{3}\s*\d{4}\s*\d{4}'

    for i in range (len(text)):
        clean_text.append(re.sub(pattern1, '', text[i]))
        clean_text.append(re.sub(pattern2, '', text[i]))
        clean_text.append(re.sub(pattern3, '', text[i]))
        clean_text.append(re.sub(pattern4, '', text[i]))
        clean_text.append(re.sub(pattern5, '', text[i]))
           
    return clean_text

In [1397]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_phones)

print("--"*50)
print("Phone Number Removal:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
Phone Number Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.4 Removing Dates**

Dates may add some contextual information, they do not carry meaningful semantic value for text classification and therefore do not meaningfully contribute to the model.

In [1398]:
def extract_dates(text):
    """
    Extract phone numbers in various formats
    Returns: list of phone numbers
    """
    clean_text = []
    
    # Pattern 1: YYYY-MM-DD
    pattern1 = r'\d{4}-\d{2}-\d{2}'
    # Pattern 2: DD-MM-YYYY or MM-DD-YYYY
    pattern2 = r'\d{2}-\d{2}-\d{4}'
    # Pattern 3: MM/DD/YYYY or DD/MM/YYYY
    pattern3 = r'\d{2}/\d{2}/\d{4}'
    # Pattern 4: January 15, 2024 or Jan 15, 2024
    pattern4 = r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s\d{1,2},\s\d{4}'
    # Pattern 5: 18 May. 2025 | 17 May 2025
    pattern5 = r'\d{1,2}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s\d{4}'
    # pattern 6: 2024.01.15
    pattern6 = r'\d{4}\.\d{2}\.\d{2}'

    for i in range (len(text)):
        clean_text.append(re.sub(pattern1, '', text[i]))
        clean_text.append(re.sub(pattern2, '', text[i]))
        clean_text.append(re.sub(pattern3, '', text[i]))
        clean_text.append(re.sub(pattern4, '', text[i]))
        clean_text.append(re.sub(pattern5, '', text[i]))
        clean_text.append(re.sub(pattern6, '', text[i]))
           
    return clean_text

In [1399]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_dates)

print("--"*50)
print("Date Removal:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
Date Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.5 Removing all the URLs**

Removing URLs from the text will reduce its overall length. In turn, this helps us retain only the most relevant content for text classification.

In [1400]:
def extract_urls(text):
    """
    Remove URLs from text while preserving everything else
    """
    url_pattern = r'https?://\S+|www\.\S+'

    clean_text = []
    # Find all matches
    for i in range (len(text)):
        clean_text.append(re.sub(url_pattern, '', text[i]))
    
    return clean_text

In [1401]:
data['full_text_clean'] = data['full_text_clean'].apply(extract_urls)

print("--"*50)
print("URL Removal:")
print("--"*50)
data 

----------------------------------------------------------------------------------------------------
URL Removal:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013\n3patchcrafts\nTh...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


#### **1.6 Replacing `\n` with Space**

Replacing `\n` with a space removes the ambiguity caused when text appears on a new line but is recorded as `\n`, resulting in a single continuous text line.

In [1402]:
# Cleaning "\n" from the cleaned full_text column
def clean_text(text):
    clean = r"[\r\n]+"

    sentence = []

    # Find all matches
    for i in range (len(text)):
        sentence.append(re.sub(clean, " ", text[i]))

    sentence = [stn for stn in sentence if stn]

    return sentence

data['full_text_clean'] = data['full_text_clean'].apply(clean_text)

print("--"*50)
print("New datastructure after cleaning the full_text columns:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after cleaning the full_text columns:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa..."
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea..."
...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...


In [1403]:
# Random index number to check how a full_text field and full_text_clean field looks like
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_clean'])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'])

----------------------------------------------------------------------------------------------------
Text Field (262) -- Total Character Length (90)
----------------------------------------------------------------------------------------------------
["Blogging | T4535 - Explore & Discover: Your Gateway to Knowledge T4535 Cooking Communication Finance Fitness Blogging Decor Marketing Health Blogging How to Start a Successful Blog Fast Posted on July 17, 2025 Blogging Quick Steps to Launching a Thriving Blog Quickly. Starting a blog can seem daunting, but with the right approach, you can set up a successful blog in no time. Whether you're looking to share your... Read More digital marketing blogging content creation SEO How to Start a Successful Blog Quickly Posted on July 16, 2025 Blogging Introduction to Blogging Success. Starting a blog can seem daunting at first, but with the right approach, you can set up a successful blog quickly. This guide will walk you through the essential... R

As you can see, we currently have a list-of-lists structure, which can be difficult to manage in the subsequent steps of the execution. Therefore, we will combine it into a single list for simplicity and easier processing.

In [1404]:
data['full_text_merge'] = data['full_text_clean'].apply(lambda x: ' '.join(x))

print("--"*50)
print("New datastructure after combining the list-of-list structure into one single list:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after combining the list-of-list structure into one single list:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",Fall arrest and work positioning harness Fall...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,Vermont Mountain Eats: Jay Peak - All Mountain...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,05/18/2020 Booking Report for Bulloch County -...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,Is Pickleball Easier than Tennis? | AllRacket ...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...","Account Executive, Auto Finance - Greater Seat..."
...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,Blog post Archives - 38th VoyageMystic Seaport...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,Kids bike pedals collection 3D Model in Bicycl...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts: February 2013 3patchcrafts The p...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,Fatal error: Uncaught mysqli_sql_exception: Ta...


In [1405]:
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_merge'])})")
print("--"*50)
print(data.loc[idx, 'full_text_merge'])

----------------------------------------------------------------------------------------------------
Text Field (39) -- Total Character Length (31529)
----------------------------------------------------------------------------------------------------
Bathroom Project | Accord Renovations | Home Renovations in Ottawa +I (6I3) 878-0944 Business Hours: Monday to Friday 9-5pm SERVICES Kitchen Renovation Bathroom Renovation Finished Basement Flooring Renovation PROJECTS TESTIMONIALS ABOUT US CONTACT US SERVICES Kitchen Renovation Bathroom Renovation Finished Basement Other Renovation PROJECTS TESTIMONIALS ABOUT US CONTACT US BATHROOM PROJECT HOME / PROJECTS / BATHROOM PROJECT Phone: +I (6I3) 878-0944 Business Hours Monday to Friday 9-5pm E-mail: About Us Accord Renovations is dedicated providing you with a level of service that is unmatched. Our attention to details and eye for design ensures an enjoyable experience. We offer high quality renovation services and take personal responsibilit

In [1406]:
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_clean'])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'])

----------------------------------------------------------------------------------------------------
Text Field (39) -- Total Character Length (60)
----------------------------------------------------------------------------------------------------
['Bathroom Project | Accord Renovations | Home Renovations in Ottawa +I (6I3) 878-0944 Business Hours: Monday to Friday 9-5pm SERVICES Kitchen Renovation Bathroom Renovation Finished Basement Flooring Renovation PROJECTS TESTIMONIALS ABOUT US CONTACT US SERVICES Kitchen Renovation Bathroom Renovation Finished Basement Other Renovation PROJECTS TESTIMONIALS ABOUT US CONTACT US BATHROOM PROJECT HOME / PROJECTS / BATHROOM PROJECT Phone: +I (6I3) 878-0944 Business Hours Monday to Friday 9-5pm E-mail: About Us Accord Renovations is dedicated providing you with a level of service that is unmatched. Our attention to details and eye for design ensures an enjoyable experience. We offer high quality renovation services and take personal responsibility

#### **1.7 Normalizing Text**

Here we will convert all the sentences in the lower case, remove extra spaces, drop all the number, and punctuation using regex expression.

In [1407]:
def normalize_text(text):
    
    pattern = rf'(?<!{curr_symbols})\b\d+(?:\.\d+)?\b(?!{curr_symbols})'
    text = re.sub(pattern, '', text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text.lower()

In [1408]:
data['full_text_merge'] = data['full_text_merge'].apply(normalize_text)

print("--"*50)
print("New datastructure after converting the full_text_merge column to lowercase:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after converting the full_text_merge column to lowercase:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",fall arrest and work positioning harness fall ...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,vermont mountain eats jay peak all mountain ma...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,booking report for bulloch county allongeorgia...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,is pickleball easier than tennis allracket ski...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...",account executive auto finance greater seattle...
...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,blog post archives 38th voyagemystic seaport t...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,kids bike pedals collection 3d model in bicycl...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts february 3patchcrafts the place t...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,fatal error uncaught mysqli_sql_exception tabl...


In [1409]:
# Sanity check for the full_text_merge column
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_merge'])})")
print("--"*50)
print(data.loc[idx, 'full_text_merge'][:100])

----------------------------------------------------------------------------------------------------
Text Field (313) -- Total Character Length (60034)
----------------------------------------------------------------------------------------------------
the welsh saints project the welsh saints project home immigrants voyages about donate resources all


In [1410]:
print("--"*50)
print(f"Text Field ({idx}) -- Total Character Length ({len(data.loc[idx, 'full_text_clean'])})")
print("--"*50)
print(data.loc[idx, 'full_text_clean'])

----------------------------------------------------------------------------------------------------
Text Field (313) -- Total Character Length (60)
----------------------------------------------------------------------------------------------------
['The Welsh Saints Project The Welsh Saints Project Home Immigrants Voyages About Donate Resources All Resources Biographies General Resources Missionary Work Personal Writings Photos Q/A ADMIN MENU Administrator View On FamilySearch Edward Phillips Vital Information Close Given Name: Edward Surname: Phillips Gender: Male Birth date: 28 March 1808 Birth place: English Spelling: Wenvoe, Glamorganshire, Wales Welsh Spelling: Gwenfo, Glamorganshire, Wales Death date: 6 October 1856 Death place: Farmington, Davis, Utah, United States Burial date: 9 October 1856 Burial place: Farmington, Davis, Utah, United States Parents and Siblings Open Father: Edward Phillips Mother: Richards, Dianah Siblings Ann Phillips Elizabeth Phillips Spouses and Child

#### **1.8 Removal of Stop Words**

These are common repeating words in a sentence, such as “a,” “the,” “he,” and “there.” These words typically carry little semantic meaning on their own, as they primarily serve grammatical functions. In this assignment, where our goal is to identify meaningful classification terms, such stopwords would dominate the corpus because they appear in nearly all sentences, thereby adding noise rather than useful information.

In [1411]:
# loading stopwords from both sklearn and nltk
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.corpus import stopwords

nltk_stopwords = set(stopwords.words('english'))
sklearn_stopwords = set(ENGLISH_STOP_WORDS)

print("--"*50)
print("Number of NLTK stopwords:",len(nltk_stopwords), "\nStopwords from NLTK:", list(nltk_stopwords)[:10], "...")
print("--"*50)
print("Number of Sklearn stopwords:",len(sklearn_stopwords), "\nStopwords from Sklearn:", list(sklearn_stopwords)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Number of NLTK stopwords: 198 
Stopwords from NLTK: ["you'd", "hadn't", "they'd", 'yours', 'than', 'does', 'he', 'this', 'on', 's'] ...
----------------------------------------------------------------------------------------------------
Number of Sklearn stopwords: 318 
Stopwords from Sklearn: ['yours', 'often', 'than', 'show', 'he', 'latter', 'un', 'get', 'neither', 'her'] ...
----------------------------------------------------------------------------------------------------


In [1412]:
# Chceking the common stopwords between both libraries
common_stopwords = nltk_stopwords.intersection(sklearn_stopwords)
print("--"*50)
print("Number of Common Stopwords:", len(common_stopwords), "\nCommon Stopwords:", list(common_stopwords)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Number of Common Stopwords: 119 
Common Stopwords: ['yours', 'than', 'he', 'this', 'on', 'is', 'her', 'can', 'nor', 'from'] ...
----------------------------------------------------------------------------------------------------


**Take Away:**

Since the default stopword list in NLTK is predefined and not intended to be modified directly in place, it offers limited flexibility for customization. In contrast, scikit-learn’s stopword list can be easily copied and extended (for example, by adding or removing specific words such as negations) to better suit a particular corpus.

Therefore, we will use the scikit-learn stopword list as the base and extend it by incorporating selected stopwords from NLTK, along with additional domain-specific terms that may negatively affect the classification context based on our domain understanding.

In [1413]:
# Chceking the common stopwords between both libraries
nltk_unique = nltk_stopwords.difference(sklearn_stopwords)
print("--"*50)
print("Number of NLTK Unique Stopwords:", len(nltk_unique), "\nNLTK Unique Stopwords:", list(nltk_unique)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Number of NLTK Unique Stopwords: 79 
NLTK Unique Stopwords: ["you'd", "hadn't", "they'd", 'does', 's', 'o', "it'd", 'did', 'd', 've'] ...
----------------------------------------------------------------------------------------------------


In [1414]:
# List of common words to be removed from full_text_clean column
common_word_list = {'followers', 'instagram', 'facebook', 'like', 'messages', 'support',
                    'sitemap', 'free', 'faq', 'twitter', 'search', 'subscribe', 'youtube', 
                    'feedback', 'linkedin', 'demo', 'menu', 'following', 'unsubscribe', 
                    'snapchat', 'tiktok', 'whatsapp', 'january', 'february', 'march', 'april', 'may','june','july','august', 'september', 'october', 'november', 'december','december december'
                    'profile', 'share', 'reddit', 'download', 'settings', 'notifications', 'qa', 'people'}

# Common phrases to remove
phrase_list = ["find out more", "join now","terms & conditions", "terms and conditions","all rights reserved","free trial", 
             "click here", "about us", "contact us","terms of service", "privacy policy", "help center", "our story","our team",
             "read more", "learn more", "get started", "download now", "sign up", "log in"]

In [1415]:
# Combining sklearn and nltk stop words and adding the common words to the stop word list
stop_words_corpus = sklearn_stopwords.union(nltk_unique).union(common_word_list).union(phrase_list)

print("--"*50)
print("Total Stop Words in the Corpus:", len(stop_words_corpus), "\nSample Stop Words:", list(stop_words_corpus)[:10], "...")
print("--"*50)

----------------------------------------------------------------------------------------------------
Total Stop Words in the Corpus: 458 
Sample Stop Words: ['yours', "hadn't", 'often', 'than', 'show', 'menu', 'does', 'he', 's', 'latter'] ...
----------------------------------------------------------------------------------------------------


In [1416]:
# function to remove stop words from the full_text_merge column
def remove_stop_words(text):
    clean_text = []
    for word in text.split():
        if word not in stop_words_corpus:
            clean_text.append(word)
    
    return ' '.join(clean_text)

In [1417]:
data['analysis_text'] = data['full_text_merge'].apply(remove_stop_words)

print("--"*50)
print("New datastructure after removing stop words from the full_text_merge column:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after removing stop words from the full_text_merge column:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge,analysis_text
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ECOMMERCE,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",fall arrest and work positioning harness fall ...,fall arrest work positioning harness fall arre...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,vermont mountain eats jay peak all mountain ma...,vermont mountain eats jay peak mountain mamas ...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,NEWS,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,booking report for bulloch county allongeorgia...,booking report bulloch county allongeorgia geo...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,BLOG,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,is pickleball easier than tennis allracket ski...,pickleball easier tennis allracket skip conten...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,OTHER,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...",account executive auto finance greater seattle...,account executive auto finance greater seattle...
...,...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,BLOG,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,blog post archives 38th voyagemystic seaport t...,blog post archives 38th voyagemystic seaport 3...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ECOMMERCE,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,kids bike pedals collection 3d model in bicycl...,kids bike pedals collection 3d model bicycle 3...
593,491.0,Jaee Oh,blog,BLOG,BLOG,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts february 3patchcrafts the place t...,3patchcrafts 3patchcrafts place works creation...
594,493.0,Jaee Oh,other,OTHER,OTHER,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,fatal error uncaught mysqli_sql_exception tabl...,fatal error uncaught mysqli_sql_exception tabl...


In [1418]:
# Sanity check for the analysis_text column with full_text_merge column and full_text_clean column
idx = np.random.randint(0, 596)
print("--"*50)
print(f"Text Field ({idx})")
print("--"*50)
print(f"Total Character Length for Analysis Ready Text Column ({len(data.loc[idx, 'analysis_text'])})")
print("--"*50)
print(f"Total Character Length for full_text_merge Column ({len(data.loc[idx, 'full_text_merge'])})")
print("--"*50)
print(f"Total Character Length for full_text_clean Column ({len(data.loc[idx, 'full_text_clean'])})-- because this column is a list of lists, the character length is much higher than the other two columns")
print("--"*50)


----------------------------------------------------------------------------------------------------
Text Field (558)
----------------------------------------------------------------------------------------------------
Total Character Length for Analysis Ready Text Column (55679)
----------------------------------------------------------------------------------------------------
Total Character Length for full_text_merge Column (63509)
----------------------------------------------------------------------------------------------------
Total Character Length for full_text_clean Column (60)-- because this column is a list of lists, the character length is much higher than the other two columns
----------------------------------------------------------------------------------------------------


_________

### **2. TF-IDF Construction**

TF-IDF stands for Term Frequency-Inverse Document Frequency (Converts text → numerical features). Here, we will focus on converting our preprocessed text corpus `(analysis_text)` into a numerical feature matrix, where each document is represented as a high-dimensional vector of TF-IDF scores.

#### **2.1 Label Understanding**

Here, we will first examine the types of labels present in the dataset. Finally, we will construct a feature representation suitable for generating TF-IDF vectors with different hyperparameter configurations.

In [1419]:
# Unique labels in the dataset
print("--"*50)
print("Unique final labels in the dataset:", list(data['manual_label_final'].unique()))
print("--"*50)
print("Unique Manual Labels clean in the dataset:", list(data['manual_label_clean'].unique()))

----------------------------------------------------------------------------------------------------
Unique final labels in the dataset: ['ECOMMERCE', 'BLOG', 'NEWS', 'OTHER', 'FORUM/DISCUSSION', 'EDUCATION', 'GOVERNMENT', 'TECHNICAL']
----------------------------------------------------------------------------------------------------
Unique Manual Labels clean in the dataset: ['ECOMMERCE', 'BLOG', 'NEWS', 'OTHER', 'FORUM/DISCUSSION', 'EDUCATION', 'GOVERNMENT', 'TECHNICAL', 'BLANK', 'SPORTS', 'RELIGIOUS', 'DIRECTORY', 'TRAVEL', 'HEALTH', 'MOTORSPORTS', 'PHOTOGRAPHY', 'ENTERTAINMENT', 'LITERATURE', 'TRAVEL VLOG', 'SOFTWARE']


In [1420]:
# Converting final labels into lowercase for uniformity
data['manual_label_final'] = data['manual_label_final'].str.lower()

print("--"*50)
print("New datastructure after converting the final labels into lowercase:")
print("--"*50)
data

----------------------------------------------------------------------------------------------------
New datastructure after converting the final labels into lowercase:
----------------------------------------------------------------------------------------------------


,page_id,assigned_to,manual_label,manual_label_clean,manual_label_final,full_text,full_text_clean,full_text_merge,analysis_text
0,655.0,Vinay Varshigan Sivakumar Jayalakshmi,ecommerce,ECOMMERCE,ecommerce,Fall arrest and work positioning harness – All...,"[Fall arrest and work positioning harness , Fa...",fall arrest and work positioning harness fall ...,fall arrest work positioning harness fall arre...
1,656.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,blog,Vermont Mountain Eats: Jay Peak - All Mountain...,[Vermont Mountain Eats: Jay Peak - All Mountai...,vermont mountain eats jay peak all mountain ma...,vermont mountain eats jay peak mountain mamas ...
2,657.0,Vinay Varshigan Sivakumar Jayalakshmi,news,NEWS,news,05/18/2020 Booking Report for Bulloch County -...,[05/18/2020 Booking Report for Bulloch County ...,booking report for bulloch county allongeorgia...,booking report bulloch county allongeorgia geo...
3,658.0,Vinay Varshigan Sivakumar Jayalakshmi,blog,BLOG,blog,Is Pickleball Easier than Tennis? | AllRacket\...,[Is Pickleball Easier than Tennis? | AllRacket...,is pickleball easier than tennis allracket ski...,pickleball easier tennis allracket skip conten...
4,661.0,Vinay Varshigan Sivakumar Jayalakshmi,other,OTHER,other,"Account Executive, Auto Finance - Greater Seat...","[Account Executive, Auto Finance - Greater Sea...",account executive auto finance greater seattle...,account executive auto finance greater seattle...
...,...,...,...,...,...,...,...,...,...
591,489.0,Jaee Oh,blog,BLOG,blog,Blog post Archives - 38th VoyageMystic Seaport...,[Blog post Archives - 38th VoyageMystic Seapor...,blog post archives 38th voyagemystic seaport t...,blog post archives 38th voyagemystic seaport 3...
592,490.0,Jaee Oh,ecommerce,ECOMMERCE,ecommerce,Kids bike pedals collection 3D Model in Bicycl...,[Kids bike pedals collection 3D Model in Bicyc...,kids bike pedals collection 3d model in bicycl...,kids bike pedals collection 3d model bicycle 3...
593,491.0,Jaee Oh,blog,BLOG,blog,3patchcrafts: February 2013\n3patchcrafts\nThe...,[3patchcrafts: February 2013 3patchcrafts The ...,3patchcrafts february 3patchcrafts the place t...,3patchcrafts 3patchcrafts place works creation...
594,493.0,Jaee Oh,other,OTHER,other,Fatal error: Uncaught mysqli_sql_exception: Ta...,[Fatal error: Uncaught mysqli_sql_exception: T...,fatal error uncaught mysqli_sql_exception tabl...,fatal error uncaught mysqli_sql_exception tabl...


In [1421]:
# Analysis ready dataframe

data_analysis = data[['page_id', 'analysis_text', 'manual_label_final']]

print("--"*50)
print("Analysis ready dataframe (Rows, Columns):\n", data_analysis.shape)
print("--"*50)

----------------------------------------------------------------------------------------------------
Analysis ready dataframe (Rows, Columns):
 (596, 3)
----------------------------------------------------------------------------------------------------


#### **2.2 TFIDF Construction**

Here, we will construct the TF-IDF DataFrame. While this implementation may not be fully optimized, our focus is on understanding how TF-IDF is built and how the feature representation appears in its raw form.

In [1422]:
# Defininfg TFIDF Vectorizer
vectorizer = TfidfVectorizer( 
    max_features= 1500,      # total features, NOT per document
    ngram_range=(1, 2),      # unigrams + bigrams
    max_df=0.90,             # ignore very common words
    min_df=2                 # ignore rare noise
)

In [1423]:
# Text Transformation using TFIDF Vectorizer
tfidf_matrix = vectorizer.fit_transform(data_analysis['analysis_text'])

In [1424]:
print("--"*50)
print("Shape of the TFIDF matrix: (Rows, Columns)")
print("--"*50)  
print(tfidf_matrix.shape)
matrix_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print("--"*50)
print("TFIDF Matrix as a DataFrame:")
print("--"*50)
matrix_df

----------------------------------------------------------------------------------------------------
Shape of the TFIDF matrix: (Rows, Columns)
----------------------------------------------------------------------------------------------------
(596, 1500)
----------------------------------------------------------------------------------------------------
TFIDF Matrix as a DataFrame:
----------------------------------------------------------------------------------------------------


,1xbet,30min,3d,3d models,ab,able,absolutely,ac,accept,access,...,yang,year,years,years ago,yes,yoga,york,young,zone,zu
0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.098723,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0
1,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.016397,0.016554,0.000000,0.022247,0.0,0.0,0.000000,0.000000,0.0
2,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0
3,0.0,0.0,0.000000,0.000000,0.0,0.021152,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.015270,0.000000,0.000000,0.0,0.0,0.000000,0.026393,0.0
4,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.025737,...,0.0,0.000000,0.023354,0.000000,0.031386,0.0,0.0,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.107194,0.000000,0.0
592,0.0,0.0,0.510343,0.689532,0.0,0.000000,0.0,0.0,0.001782,0.000000,...,0.0,0.000000,0.000000,0.000000,0.008470,0.0,0.0,0.000000,0.000000,0.0
593,0.0,0.0,0.050781,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.033890,0.239497,0.395499,0.000000,0.0,0.0,0.000000,0.000000,0.0
594,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0


In [1425]:
print(f"TF-IDF Matrix shape: {matrix_df.shape}")
print(f"Average non-zeros per document: {tfidf_matrix.nnz/tfidf_matrix.shape[0]}")

density = tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
print(f"Sparsity: {(1-density) * 100:.4f}%")


TF-IDF Matrix shape: (596, 1500)
Average non-zeros per document: 103.24328859060402
Sparsity: 93.1171%


In [1426]:
# Top vocab across all the documents 

vocab = vectorizer.get_feature_names_out() # Term names
doc_freq = np.asarray((tfidf_matrix > 0).sum(axis=0)).flatten() # Number of documents that have each terms

n = 30
top_idx = doc_freq.argsort()[::-1][:n]

print("--"*50)
print("Most frequent terms (by document count):")
print("--"*50)
for i, idx in enumerate(top_idx, 1):
    print(f"  {i}. {vocab[idx]}: {doc_freq[idx]} docs")

----------------------------------------------------------------------------------------------------
Most frequent terms (by document count):
----------------------------------------------------------------------------------------------------
  1. home: 403 docs
  2. contact: 384 docs
  3. new: 305 docs
  4. email: 260 docs
  5. privacy: 254 docs
  6. content: 232 docs
  7. policy: 231 docs
  8. time: 225 docs
  9. use: 220 docs
  10. copyright: 215 docs
  11. information: 207 docs
  12. privacy policy: 202 docs
  13. terms: 190 docs
  14. rights: 189 docs
  15. website: 184 docs
  16. news: 184 docs
  17. service: 183 docs
  18. best: 179 docs
  19. reserved: 173 docs
  20. blog: 172 docs
  21. page: 170 docs
  22. help: 164 docs
  23. site: 164 docs
  24. rights reserved: 163 docs
  25. view: 163 docs
  26. work: 162 docs
  27. skip: 161 docs
  28. services: 155 docs
  29. online: 154 docs
  30. post: 150 docs


In [1427]:
# Top unique words for each labels -- this answers the question if a feature make's sense or not?
matrix_df['Category'] = data_analysis['manual_label_final'].values

print("--"*50)
print("Adding Final Label column into our TFIDF matrix")
print("--"*50)
matrix_df

----------------------------------------------------------------------------------------------------
Adding Final Label column into our TFIDF matrix
----------------------------------------------------------------------------------------------------


,1xbet,30min,3d,3d models,ab,able,absolutely,ac,accept,access,...,year,years,years ago,yes,yoga,york,young,zone,zu,Category
0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.098723,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,ecommerce
1,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.016397,0.016554,0.000000,0.022247,0.0,0.0,0.000000,0.000000,0.0,blog
2,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,news
3,0.0,0.0,0.000000,0.000000,0.0,0.021152,0.0,0.0,0.000000,0.000000,...,0.000000,0.015270,0.000000,0.000000,0.0,0.0,0.000000,0.026393,0.0,blog
4,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.025737,...,0.000000,0.023354,0.000000,0.031386,0.0,0.0,0.000000,0.000000,0.0,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.107194,0.000000,0.0,blog
592,0.0,0.0,0.510343,0.689532,0.0,0.000000,0.0,0.0,0.001782,0.000000,...,0.000000,0.000000,0.000000,0.008470,0.0,0.0,0.000000,0.000000,0.0,ecommerce
593,0.0,0.0,0.050781,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.033890,0.239497,0.395499,0.000000,0.0,0.0,0.000000,0.000000,0.0,blog
594,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,other


In [1428]:
category_means = matrix_df.groupby('Category').mean()

for category in category_means.index:
    top = category_means.loc[category].sort_values(ascending=False).head(10)
    print(f"\nCategory: {category}")
    print(top)


Category: blog
blog        0.049218
post        0.040501
said        0.037430
comments    0.030886
time        0.028009
new         0.027790
love        0.027283
home        0.027150
email       0.026278
work        0.025404
Name: blog, dtype: float64

Category: ecommerce
cart        0.049134
add         0.048127
contact     0.036028
products    0.032372
add cart    0.032141
services    0.031535
shop        0.031134
price       0.026174
wishlist    0.025340
home        0.025139
Name: ecommerce, dtype: float64

Category: education
university     0.052129
research       0.049926
information    0.031715
library        0.030815
materials      0.026850
catalog        0.025614
school         0.024048
news           0.023613
page           0.023582
home           0.023071
Name: education, dtype: float64

Category: forum/discussion
forum       0.134640
password    0.074940
html        0.068107
login       0.061287
ago         0.059748
comments    0.056724
home        0.056288
submit      0.05

**Interpretation:**

Based on the parameters, which were chosen heuristically, we observe approximately 92.95% sparsity in the TF-IDF matrix. This means that 92.95% of the entries are zeros, while only 7.05% contain non-zero values representing meaningful term weights.

With 1,500 features across 596 documents, this results in an average of roughly 2.5 non-zero features per document (after accounting for sparsity). Although the matrix is highly sparse, as expected in text data. The inclusion of `ngram_range=(1, 2)` helps capture shared unigrams and bigrams, reducing the likelihood of completely empty document vectors and improving feature overlap across documents

#### **2.3 Grid Search**

We will now use Grid Search from sklearn to optimize our parameter and find the ones which gives us the best resutls

In [1429]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# defining pgrid parameters:
parameters = {
    'tfidf__max_features' : [1000,1500],
    'tfidf__min_df' : [2,5],
    'tfidf__max_df' : [0.85,0.90],
    'tfidf__ngram_range' : [(1,2), (2,2)]
}

# Defining small pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classification', LogisticRegression(max_iter=1000))
])

# defining Grid Search
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv = 5,
    scoring= 'accuracy',
    n_jobs= -1,
    verbose=2
)

In [1430]:
# Running the Grid Search
grid_search.fit(data_analysis['analysis_text'],data_analysis['manual_label_final'])

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(2, 2); total time=   6.0s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(2, 2); total time=   5.9s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(2, 2); total time=   6.0s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(2, 2); total time=   5.9s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(2, 2); total time=   5.9s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time=   7.9s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time=   8.0s
[CV] END tfidf__max_df=0.85, tfidf__max_features=1000, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time=   

,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'tfidf__max_df': [0.85, 0.9], 'tfidf__max_features': [1000, 1500], 'tfidf__min_df': [2, 5], 'tfidf__ngram_range': [(1, ...), (2, ...)]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [1431]:
#best model outcome
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)


Best Parameters: {'tfidf__max_df': 0.85, 'tfidf__max_features': 1500, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}
Best Score: 0.4765126050420168


#### **2.4 Optimized TFIDF Construction**

Based on the results of the above grid search we shall select the new parameter and then compare the results.

In [1432]:
# Defininfg TFIDF Vectorizer
vectorizer = TfidfVectorizer( 
    max_features= 1500,      # total features, NOT per document
    ngram_range=(1, 2),      # unigrams + bigrams
    max_df=0.85,             # ignore very common words
    min_df=2                 # ignore rare noise
)

In [1433]:
# Text Transformation using TFIDF Vectorizer
tfidf_matrix = vectorizer.fit_transform(data_analysis['analysis_text'])

In [1434]:
print(f"TF-IDF Matrix shape: {matrix_df.shape}")
print(f"Average non-zeros per document: {tfidf_matrix.nnz/tfidf_matrix.shape[0]}")

density = tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
print(f"Sparsity: {(1-density) * 100:.4f}%")


TF-IDF Matrix shape: (596, 1501)
Average non-zeros per document: 103.24328859060402
Sparsity: 93.1171%


In [1435]:
# Top vocab across all the documents 
vocab = vectorizer.get_feature_names_out() # Term names
doc_freq = np.asarray((tfidf_matrix > 0).sum(axis=0)).flatten() # Number of documents that have each terms

n = 30
top_idx = doc_freq.argsort()[::-1][:n]

print("--"*50)
print("Most frequent terms (by document count):")
print("--"*50)
for i, idx in enumerate(top_idx, 1):
    print(f"  {i}. {vocab[idx]}: {doc_freq[idx]} docs")

----------------------------------------------------------------------------------------------------
Most frequent terms (by document count):
----------------------------------------------------------------------------------------------------
  1. home: 403 docs
  2. contact: 384 docs
  3. new: 305 docs
  4. email: 260 docs
  5. privacy: 254 docs
  6. content: 232 docs
  7. policy: 231 docs
  8. time: 225 docs
  9. use: 220 docs
  10. copyright: 215 docs
  11. information: 207 docs
  12. privacy policy: 202 docs
  13. terms: 190 docs
  14. rights: 189 docs
  15. website: 184 docs
  16. news: 184 docs
  17. service: 183 docs
  18. best: 179 docs
  19. reserved: 173 docs
  20. blog: 172 docs
  21. page: 170 docs
  22. help: 164 docs
  23. site: 164 docs
  24. rights reserved: 163 docs
  25. view: 163 docs
  26. work: 162 docs
  27. skip: 161 docs
  28. services: 155 docs
  29. online: 154 docs
  30. post: 150 docs


In [1436]:
# optimized TFIDF matrix
matrix_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
matrix_df['Category'] = data_analysis['manual_label_final'].values

print("--"*50)
print("Optimized TFIDF matrix:")
print("--"*50)
matrix_df

----------------------------------------------------------------------------------------------------
Optimized TFIDF matrix:
----------------------------------------------------------------------------------------------------


,1xbet,30min,3d,3d models,ab,able,absolutely,ac,accept,access,...,year,years,years ago,yes,yoga,york,young,zone,zu,Category
0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.098723,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,ecommerce
1,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.016397,0.016554,0.000000,0.022247,0.0,0.0,0.000000,0.000000,0.0,blog
2,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,news
3,0.0,0.0,0.000000,0.000000,0.0,0.021152,0.0,0.0,0.000000,0.000000,...,0.000000,0.015270,0.000000,0.000000,0.0,0.0,0.000000,0.026393,0.0,blog
4,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.025737,...,0.000000,0.023354,0.000000,0.031386,0.0,0.0,0.000000,0.000000,0.0,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
591,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.107194,0.000000,0.0,blog
592,0.0,0.0,0.510343,0.689532,0.0,0.000000,0.0,0.0,0.001782,0.000000,...,0.000000,0.000000,0.000000,0.008470,0.0,0.0,0.000000,0.000000,0.0,ecommerce
593,0.0,0.0,0.050781,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.033890,0.239497,0.395499,0.000000,0.0,0.0,0.000000,0.000000,0.0,blog
594,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,other


In [1437]:
category_means = matrix_df.groupby('Category').mean()

for category in category_means.index:
    top = category_means.loc[category].sort_values(ascending=False).head(10)
    print(f"\nCategory: {category}")
    print(top)


Category: blog
blog        0.049218
post        0.040501
said        0.037430
comments    0.030886
time        0.028009
new         0.027790
love        0.027283
home        0.027150
email       0.026278
work        0.025404
Name: blog, dtype: float64

Category: ecommerce
cart        0.049134
add         0.048127
contact     0.036028
products    0.032372
add cart    0.032141
services    0.031535
shop        0.031134
price       0.026174
wishlist    0.025340
home        0.025139
Name: ecommerce, dtype: float64

Category: education
university     0.052129
research       0.049926
information    0.031715
library        0.030815
materials      0.026850
catalog        0.025614
school         0.024048
news           0.023613
page           0.023582
home           0.023071
Name: education, dtype: float64

Category: forum/discussion
forum       0.134640
password    0.074940
html        0.068107
login       0.061287
ago         0.059748
comments    0.056724
home        0.056288
submit      0.05

**Interpretation:**

After implementing GridSearch, we observed only a slight improvement in performance. This may be because we evaluated only 80 model fits and used logistic regression as the baseline comparison model. Expanding the hyperparameter search space and experimenting with more complex or better-performing models could potentially yield stronger results.